# cnn from scratch in numpy

forward + backward by hand for conv2d, maxpool, relu, fc. partly to make sure i actually understand what torch is doing.

In [ ]:
import numpy as np

def conv2d_forward(X, W, b, stride=1, pad=0):
    n, c, h, w = X.shape
    f, _, kh, kw = W.shape
    if pad > 0:
        Xp = np.pad(X, ((0,0),(0,0),(pad,pad),(pad,pad)), mode='constant')
    else:
        Xp = X
    out_h = (h + 2*pad - kh)//stride + 1
    out_w = (w + 2*pad - kw)//stride + 1
    out = np.zeros((n, f, out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = Xp[:, :, i*stride:i*stride+kh, j*stride:j*stride+kw]
            out[:, :, i, j] = np.tensordot(patch, W, axes=([1,2,3],[1,2,3])) + b
    return out, (X, W, b, stride, pad)

## backward (the painful one)

In [ ]:
def conv2d_backward(dout, cache):
    X, W, b, stride, pad = cache
    n, c, h, w = X.shape
    f, _, kh, kw = W.shape
    if pad > 0:
        Xp = np.pad(X, ((0,0),(0,0),(pad,pad),(pad,pad)), mode='constant')
        dXp = np.zeros_like(Xp)
    else:
        Xp = X
        dXp = np.zeros_like(X)
    dW = np.zeros_like(W)
    db = dout.sum(axis=(0,2,3))
    out_h, out_w = dout.shape[2], dout.shape[3]
    for i in range(out_h):
        for j in range(out_w):
            patch = Xp[:, :, i*stride:i*stride+kh, j*stride:j*stride+kw]
            for fi in range(f):
                dW[fi] += np.sum(patch * dout[:, fi:fi+1, i:i+1, j:j+1], axis=0)
            dXp[:, :, i*stride:i*stride+kh, j*stride:j*stride+kw] += np.tensordot(dout[:, :, i, j], W, axes=([1],[0]))
    dX = dXp[:, :, pad:pad+h, pad:pad+w] if pad > 0 else dXp
    return dX, dW, db

## sanity check vs torch

In [ ]:
import torch
import torch.nn.functional as F
x = np.random.randn(2, 3, 8, 8).astype(np.float64)
w = np.random.randn(4, 3, 3, 3).astype(np.float64)
b = np.zeros(4)
out_np, _ = conv2d_forward(x, w, b, stride=1, pad=1)
xt = torch.tensor(x, requires_grad=True)
wt = torch.tensor(w)
bt = torch.tensor(b)
out_t = F.conv2d(xt, wt, bt, stride=1, padding=1).detach().numpy()
print('diff:', np.abs(out_np - out_t).max())